# 💳 FinTrack Pro: Expense Tracker - Exploratory Data Analysis & Modeling
### End-to-End Personal Finance Intelligence for Data Analysts & Business Analysts
---
**Objective**: Analyze historical spending patterns, evaluate adherence to the 50/30/20 financial rule, detect statistical spending anomalies, and build predictive models to forecast next month's outflow.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_generator import generate_synthetic_expenses
from src.data_processor import clean_and_process_expenses
from src.analytics import ExpenseAnalytics
from src.ml_predictor import SpendingPredictor

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load and Inspect Cleaned Data

In [ ]:
data_path = os.path.join(project_root, 'data', 'expenses_dataset.csv')
if not os.path.exists(data_path):
    generate_synthetic_expenses(output_path=data_path)

df = clean_and_process_expenses(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Category-Wise Spending Analysis

In [ ]:
analytics = ExpenseAnalytics(df)
cat_summary = analytics.get_category_breakdown()
cat_summary

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=cat_summary, x='Total_Amount', y='Category', palette='mako')
plt.title('Total Outflow by Spending Category (INR)', fontsize=14, weight='bold')
plt.xlabel('Total Amount (₹)')
plt.ylabel('Category')
plt.show()

## 3. 50/30/20 Budgeting Rule Audit (Needs vs Wants vs Investments)

In [ ]:
split_df = analytics.get_need_vs_want_split()
split_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(split_df))
width = 0.35
ax.bar(x - width/2, split_df['Actual_Pct'], width, label='Actual Spend %', color='#4F46E5')
ax.bar(x + width/2, split_df['Benchmark_Pct'], width, label='Benchmark Target %', color='#94A3B8')
ax.set_xticks(x)
ax.set_xticklabels(split_df['Expense_Type'])
ax.set_ylabel('Percentage (%)')
ax.set_title('Adherence to 50/30/20 Financial Budget Rule', fontsize=12, weight='bold')
ax.legend()
plt.show()

## 4. Monthly Spending Trajectory & Burn Rate

In [ ]:
monthly_df = analytics.get_monthly_trends()
monthly_df

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(monthly_df['Year_Month'], monthly_df['Total_Spend'], marker='o', color='#2563EB', linewidth=2.5)
plt.axhline(75000, color='red', linestyle='--', label='Monthly Budget Limit (₹75,000)')
plt.title('Monthly Expenditure Momentum', fontsize=13, weight='bold')
plt.xticks(rotation=45)
plt.ylabel('Amount (₹)')
plt.legend()
plt.show()

## 5. Statistical Anomaly Detection (IQR Method)

In [ ]:
anomalies = analytics.detect_anomalies_iqr()
print(f"Detected {len(anomalies)} spending anomalies:")
anomalies[['Date', 'Category', 'Amount', 'Threshold', 'Excess_Amount', 'Description']].head(10)

## 6. Machine Learning Spend Forecasting

In [ ]:
predictor = SpendingPredictor(df)
forecast = predictor.forecast_next_month_spend(target_budget=75000)
for k, v in forecast.items():
    print(f"{k}: {v}")